# Baseline error breakdown — word-only model

Trains the Phase 1 baseline (no subword channel) and produces the same
seen/unseen word familiarity table as the subword model, so the two are
directly comparable.

Outputs `results/error_analysis_wordonly.txt` and
`results/error_analysis_raw_wordonly.json`.

## How to run

1. Settings: Accelerator = GPU T4 x2, Internet = On
2. Run all cells. About 20 minutes, most of it the 35-epoch training step.
3. Collect the two files from `/kaggle/working`.

To close the tab while it runs, use Save Version -> Save & Run All (Commit)
instead and read the output from that version afterwards.

## GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'

## Code

`--word-only` was added after the previous run, so this clones `main` fresh
rather than reusing an older copy. The assertion fails immediately if the flag
is missing, rather than after the training step.

In [ ]:
import os, shutil, subprocess
os.chdir('/kaggle/working')
if os.path.exists('project'):
    shutil.rmtree('project')
!git clone -q https://github.com/hatheem-r/project_DNN.git project
os.chdir('/kaggle/working/project')
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1

helptext = subprocess.run(['python', 'notebooks/10_analysis.py', '--help'],
                          capture_output=True, text=True).stdout
assert '--word-only' in helptext, 'clone is stale: --word-only flag missing'
print('--word-only flag present')
print('HEAD:', subprocess.run(['git', 'log', '-1', '--oneline'],
                              capture_output=True, text=True).stdout.strip())

## fastText vectors

About 460 MB, downloaded to `/kaggle/temp` so it is not saved as notebook
output. The script reads the path from `SOLD_VECTORS`.

In [ ]:
import os
os.makedirs('/kaggle/temp/embeddings', exist_ok=True)
VEC = '/kaggle/temp/embeddings/cc.si.300.vec.gz'
if not os.path.exists(VEC):
    !wget -q -O $VEC https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
os.environ['SOLD_VECTORS'] = VEC
!ls -lh $VEC

## Tests

The alignment tests check that piece lists line up with words. A break there
shifts labels against tokens without raising an error.

In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2

## Run

Trains one seed for 35 epochs, then produces the breakdown. About 14 minutes
on a T4.

In [ ]:
import os, subprocess
os.makedirs('results', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

OUT = 'results/error_analysis_wordonly.txt'
with open(OUT, 'w') as log:
    p = subprocess.Popen(
        ['python', 'notebooks/10_analysis.py', '--errors', '--word-only'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        log.write(line); log.flush()
    rc = p.wait()
print('exit code', rc)

## Check the run did what it should, and read off the two numbers

Guards against the two failures the task warns about: reusing the previous
subword checkpoint, and loading a checkpoint instead of training.

In [ ]:
import os, shutil

text = open('results/error_analysis_wordonly.txt', encoding='utf-8').read()

assert 'analysis_model_wordonly.pt' in text, 'wrong checkpoint name'
assert 'no subword channel' in text, 'did not run the word-only variant'
assert 'training one seed' in text, 'loaded a checkpoint instead of training'
print('checks passed: word-only variant, trained from scratch')

# Section 2 rows look like:
#   seen   51,819   1,398   423   602   0.7677   0.6990   0.7317
f1 = {}
for line in text.splitlines():
    parts = line.split()
    if len(parts) == 8 and parts[0] in ('seen', 'unseen'):
        f1[parts[0]] = float(parts[-1])

print()
print('SEND THIS TO THE GROUP')
print('  baseline seen-word F1   =', f1.get('seen'))
print('  baseline unseen-word F1 =', f1.get('unseen'))
if 'seen' in f1 and 'unseen' in f1:
    print('  gap                     = %.4f' % (f1['seen'] - f1['unseen']))
print('  subword model was: seen 0.7317, unseen 0.4579, gap 0.2739')

for f in ('results/error_analysis_wordonly.txt',
          'results/error_analysis_raw_wordonly.json'):
    shutil.copy(f, '/kaggle/working/')
    print('saved', f, os.path.getsize(f), 'bytes')

## What to send

    baseline seen-word F1   = ____
    baseline unseen-word F1 = ____

The comparison the paper needs is the baseline's unseen-word F1 against the
subword model's 0.4579. A low baseline number supports the subword claim; a
similar one does not.

Then put both output files in `results/` and commit:

```
git add results/error_analysis_wordonly.txt results/error_analysis_raw_wordonly.json
git commit -m "word-only baseline error breakdown"
git pull --rebase
git push
```